# LangChain Essentials: Practical Notebook

This notebook walks through core LangChain patterns using modern LCEL (`langchain-core`, `langchain-openai` / `langchain-anthropic`):
1. **Setup & Model Initialization**
2. **Prompt Templates & LCEL (LangChain Expression Language)**
3. **Few-Shot Prompting**
4. **Structured Outputs (with Pydantic)**
5. **Tool Calling**
6. **Simple In-Memory RAG (Retrieval-Augmented Generation)**

## 1. Installation & Setup

In [ ]:
# Install required packages
# !pip install -qU langchain langchain-core langchain-openai langchain-community pydantic faiss-cpu

In [ ]:
import os
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

## 2. Prompt Templates & LCEL Chains
Use `ChatPromptTemplate` and the pipe (`|`) operator to compose chains.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a concise technical assistant. Explain concepts clearly in 2-3 sentences.'),
    ('human', 'Explain {topic} to a {audience}.')
])

chain = prompt | llm | StrOutputParser()

response = chain.invoke({'topic': 'vector embeddings', 'audience': 'Python developer'})
print(response)

## 3. Few-Shot Prompting
Guide the model's output format and classification logic using input-output examples.

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {'review': 'The shipping was fast and the jacket fits great!', 'sentiment': 'Positive'},
    {'review': 'It arrived on Tuesday. The color is slightly darker than the photo.', 'sentiment': 'Neutral'},
    {'review': 'The zipper broke the first time I tried it on.', 'sentiment': 'Negative'},
]

example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'Review: {review}'),
    ('ai', '{sentiment}')
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Classify the sentiment of the customer review as Positive, Neutral, or Negative.'),
    few_shot_prompt,
    ('human', 'Review: {input}')
])

sentiment_chain = final_prompt | llm | StrOutputParser()
print(sentiment_chain.invoke({'input': 'Battery life lasts all day, though the screen could be brighter.'}))

## 4. Structured Outputs with Pydantic
Force the LLM to return validated JSON matching a Pydantic schema.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class BookSummary(BaseModel):
    title: str = Field(description='Title of the book')
    author: str = Field(description='Author of the book')
    key_takeaways: List[str] = Field(description='Top 3 actionable takeaways')

structured_llm = llm.with_structured_output(BookSummary)

result = structured_llm.invoke('Summarize the book Think Again by Adam Grant.')
print(result.model_dump_json(indent=2))

## 5. Tool Calling
Bind custom Python functions as tools that the model can decide to call.

In [ ]:
from langchain_core.tools import tool

@tool
def calculate_compound_interest(principal: float, rate: float, years: int) -> float:
    """Calculate compound interest given principal, annual rate (as decimal, e.g. 0.08), and years."""
    return round(principal * ((1 + rate) ** years), 2)

llm_with_tools = llm.bind_tools([calculate_compound_interest])

ai_msg = llm_with_tools.invoke('If I invest 10000 at 8% annual interest for 5 years, how much will I have?')
print('Tool calls:', ai_msg.tool_calls)

if ai_msg.tool_calls:
    tool_call = ai_msg.tool_calls[0]
    tool_output = calculate_compound_interest.invoke(tool_call['args'])
    print('Computed Result:', tool_output)

## 6. Simple In-Memory RAG Pipeline
Embed documents, store them in a vector store, and retrieve context to answer questions.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

docs = [
    Document(page_content='LCEL stands for LangChain Expression Language and uses the | pipe operator.'),
    Document(page_content='Pydantic models can be passed to .with_structured_output() for schema validation.'),
    Document(page_content='LangGraph is used for stateful, multi-actor cyclic agent workflows.')
]

vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

rag_prompt = ChatPromptTemplate.from_template(
    'Answer the question based only on the following context:\n{context}\n\nQuestion: {question}'
)

def format_docs(docs):
    return '\n'.join(d.page_content for d in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke('What is LCEL?'))